# 00 · 纯云端高速同步 MiniMax H3 满血高配到 5TB Google Drive

**为什么这一步选免费 CPU（0 CU）？**
- 本步只是通过 Google 机房千兆内网把模型从 Hugging Face 直接写入你的 **5TB Google Drive (`/MyDrive/AI_Video/models/`)**，完全用不到 GPU！
- 用免费 CPU 跑下载速度一模一样，**省下全部 A100 80GB 计算单元专门留给 01/02 号 Notebook 生成视频！**

**A100 80GB 默认满血高配（33.1B 全参数未剪枝无损版，总计约 159 GB，仅占 5TB 空间 3.1%）**：
1. `minimax_h3_fl2va_bf16`（33.1B 未剪枝无损 BF16 ~61.7GB + Qwen3-VL-32B INT8-ConvRot ~27GB + FP16 视频 VAE + FP32 音频 VAE）
2. `minimax_h3_ref2va_bf16`（33.1B 未剪枝无损 BF16 全能多参考，复用编码器与双 VAE 仅增 ~61.7GB）
3. `minimax_h3_turbo_lora`（4~8 步加速 LoRA ~2GB）

In [ ]:
#@title 1. 挂载 5TB Google Drive 并拉取最新仓库代码
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

!pip install -q "huggingface_hub[hf_transfer]"
!python {REPO_DIR}/scripts/sync_models_to_drive.py --manifest {REPO_DIR}/configs/models_manifest.json --packs dummy --list


In [ ]:
#@title 2. 勾选同步到 5TB Google Drive 的 MiniMax H3 满血模型包（默认已选 A100 80GB 最高配）
MINIMAX_H3_FL2VA_BF16_FULL = True #@param {type:"boolean"}
MINIMAX_H3_REF2VA_BF16_FULL = True #@param {type:"boolean"}
MINIMAX_H3_TURBO_LORA = True #@param {type:"boolean"}
MINIMAX_H3_FL2VA_INT8_FULL = False #@param {type:"boolean"}
MINIMAX_H3_REF2VA_INT8_FULL = False #@param {type:"boolean"}
QWEN3VL_BF16_OPTIONAL = False #@param {type:"boolean"}
FLUX2_KEYFRAME = False #@param {type:"boolean"}
FORCE_REDOWNLOAD = False #@param {type:"boolean"}
HF_TOKEN_INPUT = "" #@param {type:"string"}

import os, subprocess

hf_token = HF_TOKEN_INPUT.strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        hf_token = ''

packs = []
if MINIMAX_H3_FL2VA_BF16_FULL: packs.append('minimax_h3_fl2va_bf16')
if MINIMAX_H3_REF2VA_BF16_FULL: packs.append('minimax_h3_ref2va_bf16')
if MINIMAX_H3_TURBO_LORA: packs.append('minimax_h3_turbo_lora')
if MINIMAX_H3_FL2VA_INT8_FULL: packs.append('minimax_h3_fl2va_int8_full')
if MINIMAX_H3_REF2VA_INT8_FULL: packs.append('minimax_h3_ref2va_int8_full')
if QWEN3VL_BF16_OPTIONAL: packs.append('qwen3vl_bf16_optional')
if FLUX2_KEYFRAME: packs.append('flux2_keyframe')

if not packs:
    raise ValueError('请至少勾选一个模型包！')

cmd = [
    'python', f'{REPO_DIR}/scripts/sync_models_to_drive.py',
    '--manifest', f'{REPO_DIR}/configs/models_manifest.json',
    '--packs', *packs,
]
if FORCE_REDOWNLOAD:
    cmd.append('--force')
if hf_token:
    cmd.extend(['--hf-token', hf_token])

print('执行命令:', ' '.join(cmd[:-2] + ['--hf-token', '***'] if hf_token else cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 3. 检查 5TB Google Drive 中已就绪的模型清单与磁盘占用
!du -sh /content/drive/MyDrive/AI_Video/models/*
!find /content/drive/MyDrive/AI_Video/models -type f -exec ls -lh {} +
